In [ ]:
# Install dependencies if needed
# %pip install -r ../requirements.txt
# dbutils.library.restartPython()

In [ ]:
import sys
import time
from pprint import pprint

sys.path.insert(0, '..')

from multiAgentSystem.agents.analyzer import analyzer_node
from multiAgentSystem.config import DEFAULT_KEYWORDS
from experiments.mlflow_setup import (
    setup_experiment,
    create_agent_run,
    log_agent_metrics,
    log_state_snapshot,
    enable_autologging
)
from experiments.mock_data import ANALYZER_TEST_STATES, get_test_state

print("✓ Imports successful")
print(f"Default keywords: {DEFAULT_KEYWORDS}")

In [ ]:
# Setup MLflow experiment
enable_autologging()
experiment_id = setup_experiment("analyzer")
print(f"Experiment ID: {experiment_id}")

In [ ]:
def run_analyzer_test(scenario_name: str, verbose: bool = True):
    """
    Run a single analyzer test scenario with MLflow tracking.
    """
    scenario = get_test_state("analyzer", scenario_name)
    test_state = scenario["state"].copy()
    expected = scenario["expected"]
    
    with create_agent_run("analyzer", scenario=scenario_name) as run:
        log_state_snapshot(test_state, prefix="input")
        
        start_time = time.time()
        try:
            result = analyzer_node(test_state)
            success = True
        except Exception as e:
            result = {"error": str(e)}
            success = False
        latency_ms = (time.time() - start_time) * 1000
        
        # Calculate new keywords
        output_keywords = result.get("keywords", [])
        input_keywords = test_state.get("keywords", [])
        new_keywords = [k for k in output_keywords if k not in input_keywords]
        
        # Check if DEFAULT_KEYWORDS are included
        has_default_keywords = all(k in output_keywords for k in DEFAULT_KEYWORDS[:3])
        
        log_agent_metrics(
            latency_ms=latency_ms,
            success=success,
            additional_metrics={
                "input_keywords_count": len(input_keywords),
                "output_keywords_count": len(output_keywords),
                "new_keywords_count": len(new_keywords),
                "has_default_keywords": 1.0 if has_default_keywords else 0.0,
                "analyze_parse_loops": result.get("analyze_parse_loops", 0),
            }
        )
        
        log_state_snapshot(result, prefix="output")
        
        # Check expected outcomes
        passed = True
        if expected.get("has_keywords") and not output_keywords:
            passed = False
        if expected.get("includes_default_keywords") and not has_default_keywords:
            passed = False
        if expected.get("minimal_new_keywords") and len(new_keywords) > 3:
            passed = False
        
        import mlflow
        mlflow.log_metric("test_passed", 1.0 if passed else 0.0)
        
        if verbose:
            status = "✅ PASSED" if passed else "❌ FAILED"
            print(f"\n{status} - {scenario_name}")
            print(f"  Description: {scenario['description']}")
            print(f"  Latency: {latency_ms:.2f}ms")
            print(f"  Input keywords: {len(input_keywords)}")
            print(f"  Output keywords: {len(output_keywords)}")
            print(f"  New keywords: {new_keywords[:5]}{'...' if len(new_keywords) > 5 else ''}")
            print(f"  Has default keywords: {has_default_keywords}")
            print(f"  analyzer_satisfied: {result.get('analyzer_satisfied')}")
        
        return result, passed, latency_ms

## Test 1: Initial Keywords

Generate initial keywords from hypotheses. Should include DEFAULT_KEYWORDS.

In [ ]:
result_1, passed_1, latency_1 = run_analyzer_test("initial_keywords")
print(f"\nGenerated keywords: {result_1.get('keywords', [])}")

## Test 2: Refined Keywords

Refine keywords based on previous GC-related findings.

In [ ]:
result_2, passed_2, latency_2 = run_analyzer_test("refined_keywords")
print(f"\nGenerated keywords: {result_2.get('keywords', [])}")

## Test 3: Convergence Test

When comprehensive keywords exist, minimal new keywords should be generated.

In [ ]:
result_3, passed_3, latency_3 = run_analyzer_test("convergence_test")
print(f"\nGenerated keywords: {result_3.get('keywords', [])}")

## Summary

In [ ]:
print("=" * 60)
print("ANALYZER AGENT TEST SUMMARY")
print("=" * 60)

tests = [
    ("initial_keywords", passed_1, latency_1),
    ("refined_keywords", passed_2, latency_2),
    ("convergence_test", passed_3, latency_3),
]

total_passed = sum(1 for _, passed, _ in tests if passed)
avg_latency = sum(lat for _, _, lat in tests) / len(tests)

for name, passed, latency in tests:
    status = "✅" if passed else "❌"
    print(f"  {status} {name}: {latency:.2f}ms")

print("=" * 60)
print(f"Total: {total_passed}/{len(tests)} passed")
print(f"Average latency: {avg_latency:.2f}ms")
print("=" * 60)